# Medical Insurance Cost Prediction

**Regression Project 52**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict individual medical insurance charges based on personal attributes such as age, BMI,
smoking status, number of children, and region. This dataset is small but rich in interpretable
patterns — especially the dramatic effect of smoking on insurance costs.

This project teaches how categorical and continuous features interact in regression and why
domain knowledge matters for feature interpretation.

## 3. Learning Objectives

1. Build a regression pipeline for healthcare cost prediction
2. Handle mixed feature types (numeric + categorical)
3. Discover the dominant effect of smoking on insurance costs
4. Use interaction features (smoker × BMI) to capture non-linear patterns
5. Evaluate with RMSE, MAE, R² and residual analysis
6. Compare simple baselines against gradient boosting and AutoML

## 4. Problem Statement

> Given personal attributes (age, sex, BMI, children, smoker, region), predict **medical insurance charges**.

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Insurance companies | Premium pricing and risk assessment |
| Policyholders | Understanding cost drivers |
| Healthcare planners | Population cost forecasting |
| Regulators | Fair pricing auditing |
| ML learners | Classic small-data regression with clear feature effects |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `mirichoi0218/insurance` |
| Records | 1,338 |
| Features | 6 (age, sex, bmi, children, smoker, region) |
| Target | `charges` (continuous, USD) |
| Key insight | Smoking status is the single strongest predictor |

## 7. Dataset Source and License Notes

- **Kaggle**: [mirichoi0218/insurance](https://www.kaggle.com/datasets/mirichoi0218/insurance)
- Originally from a textbook on ML with R
- License: Database Contents License (DbCL)
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'mirichoi0218/insurance'
TARGET = 'charges'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 90
MAX_ROWS = 50000

## 11. Dataset Download or Loading

We download the insurance dataset via `kagglehub`. Requires Kaggle credentials.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/mirichoi0218/insurance'
    )
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV files found in {path}'
df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

Verify target column, missing values, duplicates, and data types.

In [ ]:
assert TARGET in df_raw.columns, f"Target '{TARGET}' not found"
print(f'Missing values:\n{df_raw.isnull().sum()}\n')
print(f'Duplicated rows: {df_raw.duplicated().sum()}')
df = df_raw.copy()
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
print(f'Shape after cleaning: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')

## 13. Exploratory Data Analysis

With only 6 features, we can explore each one thoroughly.

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, ['age', 'bmi', 'children', 'charges']):
    if col in df.columns:
        df[col].hist(bins=30, ax=ax, alpha=0.7, color='steelblue')
        ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
cat_cols_eda = ['sex', 'smoker', 'region']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cat_cols_eda):
    if col in df.columns:
        df.groupby(col)[TARGET].mean().plot.bar(ax=ax, color='coral')
        ax.set_title(f'Mean {TARGET} by {col}')
        ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Smoker vs non-smoker charge distribution
if 'smoker' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, grp in df.groupby('smoker'):
        grp[TARGET].hist(bins=30, ax=ax, alpha=0.5, label=f'smoker={label}')
    ax.legend(); ax.set_title('Charges by Smoker Status'); plt.show()

In [ ]:
num_cols_eda = df.select_dtypes(include=[np.number]).columns.tolist()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[num_cols_eda].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.show()

## 14. Target Analysis

Insurance charges are heavily right-skewed with a multi-modal pattern driven by smoking status.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=50, color='coral', alpha=0.7)
axes[0].set_title('Charges Distribution')
axes[1].hist(np.log1p(df[TARGET]), bins=50, color='seagreen', alpha=0.7)
axes[1].set_title('log(Charges) Distribution')
plt.tight_layout(); plt.show()
print(f'Mean: ${df[TARGET].mean():,.0f}')
print(f'Median: ${df[TARGET].median():,.0f}')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split. Dataset is small (1,338 rows) so each split is modest.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- **Numeric** (age, bmi, children): Impute + StandardScaler
- **Categorical** (sex, smoker, region): Impute + OneHotEncoder
- No missing values in this dataset, but we keep imputers for robustness

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

## 17. Feature Engineering

Key domain-driven features:
- **smoker_bmi**: interaction between smoker status and BMI (smokers with high BMI have extreme costs)
- **age_squared**: captures non-linear age effect
- **age_bin**: discretized age for pattern discovery

In [ ]:
def engineer_features(d):
    d = d.copy()
    if 'bmi' in d.columns and 'smoker' in d.columns:
        smoker_flag = (d['smoker'] == 'yes').astype(int) if d['smoker'].dtype == object else d['smoker']
        d['smoker_bmi'] = smoker_flag * d['bmi']
    if 'age' in d.columns:
        d['age_squared'] = d['age'] ** 2
        d['age_bin'] = pd.cut(d['age'], bins=[0, 25, 40, 55, 100], labels=False)
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
# Rebuild preprocessor
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

Three baselines: DummyRegressor (mean), LinearRegression, and RandomForest.

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE=${r['RMSE']:,.0f}, MAE=${r['MAE']:,.0f}")

## 19. LazyPredict Benchmark

Quick comparison of ~30 regression models.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML AutoML with 90-second budget.

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE=${results['FLAML']['RMSE']:,.0f}")

## 21. Top 3 Model Selection

We select three gradient boosting variants based on LazyPredict/FLAML rankings.

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=SEED)
print(f'Top 3 models: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

Train on full training set, evaluate on held-out test set.

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE=${final[name]['RMSE']:,.0f}, MAE=${final[name]['MAE']:,.0f}")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.4, s=15)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.4, s=15)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

Examining where the best model struggles most.

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: ${abs_errors.mean():,.0f}')
print(f'Median absolute error: ${np.median(abs_errors):,.0f}')
print(f'90th percentile error: ${np.percentile(abs_errors, 90):,.0f}')
print(f'Max error: ${abs_errors.max():,.0f}')

## 24. Interpretation and Business Insight

- **Smoking** is the dominant cost driver — smokers pay 3–4x more on average
- **BMI** interacts strongly with smoking — obese smokers have the highest costs
- **Age** has a positive linear relationship with charges
- **Region** and **sex** have relatively minor effects
- **Children** count has a small positive effect
- The `smoker_bmi` interaction feature captures the most important non-linearity

## 25. Limitations

1. Very small dataset (1,338 rows) limits model complexity
2. No pre-existing conditions, medication use, or lifestyle details
3. Synthetic/textbook nature limits real-world applicability
4. No temporal information (year, policy duration)
5. Region is coarse (4 US regions only)

## 26. How to Improve This Project

1. More detailed health features (diagnosis codes, prescription history)
2. Polynomial features for age and BMI
3. Target transformation (log) for better residual behavior
4. Cross-validation for more reliable estimates on small data
5. Regularized linear models with interaction terms

## 27. Production Considerations

- Real-time premium quoting engine
- Regulatory compliance for pricing factors
- Model explainability for customers (why this premium?)
- Fairness auditing (protected attributes)
- Regular recalibration with new claims data

## 28. Common Mistakes

1. Ignoring the smoker effect — it dominates everything
2. Not creating smoker × BMI interaction
3. Using R² without checking residual patterns
4. Over-engineering on a 1,338-row dataset
5. Not discussing ethical implications of health-based pricing

## 29. Mini Challenge / Exercises

1. Try log-transforming `charges` and compare results
2. Build separate models for smokers and non-smokers
3. Add a BMI category feature (underweight/normal/overweight/obese)
4. Use 5-fold cross-validation and report mean ± std R²
5. Test whether removing `region` improves or hurts performance

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict insurance charges |
| Dataset | Medical Insurance (1,338 records) |
| Difficulty | Easy–Medium |
| Key insight | Smoking is the dominant cost driver |
| Best approach | Gradient boosting with smoker×BMI interaction |
| Primary metric | R² |
| Baseline comparison | Interaction features massively improve linear models |